# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate record sets, their `@id`s, and their available fields and columns. Entities are referenced by their `@id` for robustness.

In [ ]:
print("Record sets in this dataset:")
for rs in dataset.record_sets:
    print(f"  RecordSet: {rs['@id']}  (name: {rs.get('name','')})")
    # List fields in the record set
    print("    Fields (@id and name):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"      - {field['@id']}: {field.get('name', '')}")
        else:
            # field is likely a string @id, look up full field obj
            field_obj = dataset.lookup(field)
            print(f"      - {field_obj['@id']}: {field_obj.get('name', '')}")
    # List columns (if available)
    if rs.get('column'):
        print("    Columns (@id and name):")
        for col in rs['column']:
            if isinstance(col, dict):
                print(f"      - {col['@id']}: {col.get('name', '')}")
            else:
                col_obj = dataset.lookup(col)
                print(f"      - {col_obj['@id']}: {col_obj.get('name','')}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. 

Identify and load all available record sets using their `@id`.

In [ ]:
# Gather all record set @id's for batch extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")

# For demonstration, select the first available record set
if len(record_set_ids) > 0:
    example_record_set_id = record_set_ids[0]
    print(f"\nColumns for {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

The following cell demonstrates filtering and normalization on a numeric field by referencing columns using their `@id`.

In [ ]:
# Example: select a record set and numeric field for analysis
import numpy as np

# Please adjust these @id's based on the output above (example defaults are given)
record_set_id = example_record_set_id
# Inspect the columns of the DataFrame to choose a numeric field
numeric_candidates = dataframes[record_set_id].select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields available in {record_set_id}: {numeric_candidates}")

# Pick the first numeric field available
if len(numeric_candidates) > 0:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = None

if numeric_field_id:
    threshold = dataframes[record_set_id][numeric_field_id].quantile(0.5)
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Check for a grouping field (categorical)
    object_fields = dataframes[record_set_id].select_dtypes(include=[object]).columns.tolist()
    group_field = object_fields[0] if object_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for analysis in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This section demonstrates basic distribution and relationship plots using matplotlib/seaborn for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=dataframes[record_set_id], x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have loaded the dataset as defined via its Croissant schema, examined its structure, loaded records from its record sets using their `@id`, performed simple exploratory statistical and normalization tasks on numeric fields, and visualized distributions and groupings.

**Next steps** could include more advanced statistical analysis, data cleaning steps tailored to domain context, or building machine learning models using the extracted and processed features.